# 🧹 Exercise 02 — Explore & Clean

**Day 1 · Guided**

Profile a DataFrame, fix dtypes, handle missing values, remove duplicates, and save a clean subset.

---

In [ ]:
import pandas as pd
import geopandas as gpd

# TODO: Replace with your city slug (same as Exercise 01)
CITY_SLUG = "your_city"  # <-- change this!

# Load from saved GeoPackage
gdf = gpd.read_file(f"../data/raw_{CITY_SLUG}.gpkg")

# Extract lat/lon from geometry and convert to regular DataFrame
gdf["lat"] = gdf.geometry.centroid.y
gdf["lon"] = gdf.geometry.centroid.x
df = pd.DataFrame(gdf.drop(columns=["geometry"]))

# Rename osmid → id, drop osmnx metadata
if "osmid" in df.columns:
    df = df.rename(columns={"osmid": "id"})
for col in ["nodes", "ways"]:
    if col in df.columns:
        df = df.drop(columns=[col])

print(f"Loaded {len(df)} rows, {len(df.columns)} columns")

### 2.1 📊 Profile the data

Before fixing anything, understand what you have. Three best friends: `.info()`, `.describe()`, `.dtypes`.

In [ ]:
# .info() gives you: columns, non-null counts, dtypes, memory usage
df.info()

In [ ]:
# TODO: Run .describe() — what does it tell you about lat/lon?
# Does the range make sense for your city?

In [ ]:
# TODO: Check the dtypes of lat and lon.
# Are they float64? If not, that's a problem we need to fix.
# Hint: df["lat"].dtype

**Think:** How many columns? What % non-null? Are `lat`/`lon` float64? Does the coordinate range match your district?

### 2.2 🔧 Fix data types

Coordinates must be numeric. In some exports they arrive as strings.

In [ ]:
# Ensure lat/lon are float — pd.to_numeric handles strings → float
# errors="coerce" turns unparseable values into NaN instead of crashing
df["lat"] = pd.to_numeric(df["lat"], errors="coerce")
df["lon"] = pd.to_numeric(df["lon"], errors="coerce")

print(f"lat dtype: {df['lat'].dtype}")
print(f"lon dtype: {df['lon'].dtype}")

# Drop rows where conversion failed
before = len(df)
df = df.dropna(subset=["lat", "lon"])
print(f"Dropped {before - len(df)} rows with invalid coordinates")

### 2.3 🕳️ Handle missing values

Most tag columns are >90% empty — normal for OSM, not every node has every tag.

In [ ]:
# Calculate fill rate for each column
fill_rate = (df.notna().sum() / len(df) * 100).sort_values(ascending=False)

print("Top 20 columns by fill rate:")
print(fill_rate.head(20).round(1).to_string())

print(f"\nColumns with <1% fill rate: {(fill_rate < 1).sum()}")
print(f"Columns with <5% fill rate: {(fill_rate < 5).sum()}")

In [ ]:
# TODO: Drop columns with very low fill rate (<1%)
# But keep important columns even if sparse: cuisine, opening_hours, phone, website, wheelchair

# Hint:
# sparse_cols = fill_rate[fill_rate < 1].index.tolist()
# keep_anyway = {"cuisine", "opening_hours", "phone", "website", "wheelchair"}
# drop_cols = [c for c in sparse_cols if c not in keep_anyway]
# df = df.drop(columns=drop_cols)

### 2.4 🔁 Remove duplicates

Same coordinates but different IDs — happens from mass imports or editing mistakes.

In [ ]:
# TODO: How many duplicate coordinates are there?
# Hint: df.duplicated(subset=["lat", "lon"], keep=False).sum()

In [ ]:
# TODO: Look at some duplicates — are they truly the same place or different things at the same spot?
# Hint:
# dupes = df[df.duplicated(subset=["lat", "lon"], keep=False)]
# dupes.sort_values(["lat", "lon"]).head(10)[["id", "lat", "lon", "amenity", "name"]]

In [ ]:
# TODO: Remove duplicates (keep first occurrence)
# Hint:
# before = len(df)
# df = df.drop_duplicates(subset=["lat", "lon"], keep="first")
# print(f"Removed {before - len(df)} duplicates")

**Think:** We used `keep="first"` — is that always right? What if the second duplicate has better tag data?

### 2.5 🏷️ Extract a primary name

OSM has `name`, `name:en`, `name:sv`, `name:ar`… We need one usable column.

In [ ]:
# What name columns do we have?
name_cols = [c for c in df.columns if c.startswith("name")]
print(f"Name-related columns: {sorted(name_cols)}")

In [ ]:
# TODO: Create a 'primary_name' column.
# Priority: name > name:en > first available name:* column
# If no name is available, leave it as None.

# Hint: use df.apply() with a function that checks each column in order

### 2.6 💾 Select core columns & save

In [ ]:
# TODO: Create a clean DataFrame with these core columns.
# Keep: id, lat, lon, amenity, primary_name, cuisine, opening_hours
# Also keep: phone, website, wheelchair, addr:street, addr:housenumber (if they exist)

# Hint:
# core_cols = ["id", "lat", "lon", "amenity", "primary_name", "cuisine", "opening_hours"]
# optional_cols = ["phone", "website", "wheelchair", "addr:street", "addr:housenumber"]
# available = [c for c in optional_cols if c in df.columns]
# df_clean = df[core_cols + available].copy()

In [ ]:
# TODO: Save your clean DataFrame
# Hint: df_clean.to_csv(f"../data/clean_{CITY_SLUG}.csv", index=False)

---

✅ Correct dtypes · ✅ No duplicate coordinates · ✅ Sparse columns removed · ✅ `primary_name` column · ✅ Clean CSV saved

#### Checklist
- [ ] `df_clean.dtypes` → `lat`/`lon` are `float64`
- [ ] `df_clean.duplicated(subset=['lat', 'lon']).sum()` → `0`
- [ ] `df_clean['primary_name'].notna().mean()` → > 0.5

**Next →** [Exercise 03 — Tag Normalization](03_tag_normalization.ipynb)